# 1. Find the Order — Baseline solution

A **deterministic** baseline. It uses `prefix.json` — the two chunks known to come
first — and leaves every other chunk in the order it was given.

It writes **`answers.json`** at the repo root in the rank convention: `P[i]` is the
predicted chronological position of `chunk_i.wav`.

Replace the logic below with your own.

In [1]:
import os, json
from datasets import load_dataset
# assumes that the dataset https://huggingface.co/datasets/IOAI-official/ioai-2026-find-the-order is fully downloaded
# using git clone and then git lfs pull
# into the current directory and the dataset directory is already renamed 'dataset'
LB = 'b' # select leaderboard a or b for evaluation
TEST_DIR = f"dataset/private/test_leaderboard_{LB}"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

# prefix.json ships with every split. answers.json does NOT exist in the hidden
# grading set -- never read it from TEST_DIR, or your notebook dies at grade time.
with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefix = json.load(f)

def n_chunks(ddir):
    return sum(1 for f in os.listdir(ddir)
               if f.startswith("chunk_") and f.endswith(".wav"))

# Only numeric directories are dialogues -- skip strays such as .ipynb_checkpoints,
# which JupyterLab creates as soon as you open something inside the folder.
dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR)
     if d.isdigit() and os.path.isdir(os.path.join(TEST_DIR, d))),
    key=int,
)

answers = {}
for did in dialogue_ids:
    n = n_chunks(os.path.join(TEST_DIR, did))
    first, second = prefix[did]
    order = [first, second] + [i for i in range(n) if i not in (first, second)]
    rank = [0] * n                      # rank[i] = position of chunk_i
    for pos, idx in enumerate(order):
        rank[idx] = pos
    answers[did] = rank

with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")
# --------------------------------------------------------------------------- #
# metric (self-contained port of metrics/pairwise_accuracy.py)
# --------------------------------------------------------------------------- #
def validate_permutation(perm, n) -> bool:
    """True iff perm is a permutation of 0..n-1 (0-indexed, each exactly once)."""
    if not isinstance(perm, list) or len(perm) != n:
        return False
    seen = [False] * n
    for x in perm:
        if not isinstance(x, int) or isinstance(x, bool):
            return False
        if x < 0 or x >= n or seen[x]:
            return False
        seen[x] = True
    return True


def score_one(pred, target) -> float:
    """Pairwise ordering accuracy for RANK-convention permutations.

    Both `pred` and `target` are rank arrays: value[i] = chronological position
    of chunk i. A pair (i, j) is correct iff pred and target agree on the sign of
    (value[i] - value[j]). Ranks are distinct, so signs are never zero. O(n^2),
    which is trivial for n <= 20 and avoids the order-vs-rank convention trap of a
    generic inversion counter.
    """
    n = len(target)
    if n < 2:
        return 1.0
    if not validate_permutation(pred, n):
        return 0.0
    discordant = 0
    total = n * (n - 1) // 2
    for i in range(n):
        for j in range(i + 1, n):
            if (target[i] > target[j]) != (pred[i] > pred[j]):
                discordant += 1
    return 1.0 - discordant / total
preds = json.load(open('answers.json'))
gt = json.load(open(f'dataset/private/test_leaderboard_{LB}_answers.json'))
scores = []
for item in preds.keys():
    scores.append(score_one(preds[item], gt[item]))
print(sum(scores) / len(scores))

wrote answers.json: 100 dialogues
0.6914595881414768


## Additional Models & Libraries

## wav2-vec2-base-960h

In [2]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor

WAV2VEC_PATH = "facebook/wav2vec2-base-960h"
SAMPLE_AUDIO = "dataset/public/train/0/chunk_0.wav"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)
model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH).to(device).eval()

audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    h = model(inputs.input_values.to(device)).last_hidden_state
embed = h.mean(dim=1).squeeze().cpu().numpy()

print(f"Wav2vec embedding shape: {embed.shape}")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2vec embedding shape: (768,)


# 2. Crude Improvement on the Baseline

This pipeline, a **wav2vec2-based reranking** solution, applies supervised regression on audio embeddings to determine total chronological sequence order:

* **Embedding Extraction:** Audio files are processed through a Wav2Vec2 backbone to extract average-pooled frame embeddings for each chunk as shown in the loading code above (provided to all contestants in the baseline file).

* **Relative Position Regression:** A Ridge regression model ($\alpha=10.0$) is trained on public training pairs to directly regress the relative position of each audio file based on the embeddings of each audio file.

* **Prefix Constraints & Reranking:** On the target set, predicted float scores are computed across all chunks. Known initial segments specified in `prefix.json` are manually assigned minimal priority scores (`-1.0` and `-0.5`), after which `np.argsort` translates raw regression outputs into discrete integer rank orderings.

In [3]:
# baseline reranking method: direct regression of relative position using wav2vec embeddings
# first, collect training data
import os
from tqdm import tqdm
train_dirs = [x for x in os.listdir('dataset/public/train') if '.' not in x]
train_labels = json.load(open('dataset/public/train_answers.json'))
embeds, labels = [], []
for item in tqdm(train_dirs):
    entries = 'dataset/public/train/' + item
    audio_files = [entries + '/' + x for x in os.listdir(entries) if ".wav" in x]
    for fn in audio_files:
        audio, _ = librosa.load(fn, sr=16000)
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            h = model(inputs.input_values.to(device)).last_hidden_state
        embed = h.mean(dim=1).squeeze().cpu().numpy()
        embeds.append(embed)
        labels.append(train_labels[item][int(fn.split('_')[-1].split('.')[0])] / len(audio_files))
        

100%|██████████| 1288/1288 [03:32<00:00,  6.06it/s]


In [4]:
from sklearn.linear_model import Ridge, LinearRegression
from scipy.stats import kendalltau
from sklearn.model_selection import KFold
import numpy as np
X = np.array(embeds)
y = np.array(labels)
kf = KFold(n_splits=5, shuffle=False)
for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    reg = Ridge(alpha=10.0)
    reg.fit(X_train, y_train)
    preds = reg.predict(X_test)
    tau, _ = kendalltau(y_test, preds)
    print(tau)
reg = Ridge(alpha=10.0)
reg.fit(X, y) # this isn't the most rigorous of evaluations
# the most rigorous evaluations should involve splitting based on strict conversation boundaries and calculating
# competition metric after applying this wav2vec2-based reranker on top of the prefix.json baseline
# but this is what i used during the competition, it saves time and at least reassures that the reranking
# is of nontrivial quality (kendall's tau > 0.15)

0.19028940543669873
0.17172099071904112
0.19318691651583358
0.19020431766878712
0.1785177359306407


,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",10.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [5]:
import os
from tqdm import tqdm
dirs = [x for x in os.listdir(f'dataset/private/test_leaderboard_{LB}') if '.' not in x]
prefixes = json.load(open(f'dataset/private/test_leaderboard_{LB}/prefix.json'))
#train_labels = json.load(open('dataset/private/test_leaderboard_a_answers.json'))
answers = {}
for item in tqdm(dirs):
    entries = f'dataset/private/test_leaderboard_{LB}/' + item
    audio_files = sorted([entries + '/' + x for x in os.listdir(entries) if ".wav" in x])
    # remember to get all the sorting steps right :skull:
    temp_embeds = []
    for fn in audio_files:
        audio, _ = librosa.load(fn, sr=16000)
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            h = model(inputs.input_values.to(device)).last_hidden_state
        embed = h.mean(dim=1).squeeze().cpu().numpy()
        temp_embeds.append(embed)
    ranker_outputs = reg.predict(np.array(temp_embeds))
    #print(ranker_outputs)
    fn_prefix = prefixes[item]
    # apply prefix
    ranker_outputs[fn_prefix[0]] = -1.0
    ranker_outputs[fn_prefix[1]] = -0.5
    answers[item] = [int(x) for x in list(np.argsort(np.argsort(ranker_outputs)))]
    #print(kendalltau(answers[item], train_labels[item]))
        

100%|██████████| 100/100 [00:18<00:00,  5.43it/s]


In [6]:
with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")

wrote answers.json: 100 dialogues


In [126]:
preds = json.load(open('answers.json'))
gt = json.load(open(f'dataset/private/test_leaderboard_{LB}_answers.json'))
#gt = json.load(open('dataset/public/train_answers.json'))
scores = []
for item in preds.keys():
    scores.append(score_one(preds[item], gt[item]))
print(sum(scores) / len(scores))
# 0.7110 score (lb A)
# 0.7095 score (lb B) :) 
# the gain is less than expected (from my tests if the advantages come uniformly the score should go up to 0.72+)
# that means the wav2vec2 + ridge regressor is probably better at ranking stuff near the beginning than the average audio file
# next solution approach to try: whisper speech-to-text + qwen2.5-0.5b perplexity based 
# greedy / beam-search sorting


0.7095114406577255


## Qwen2.5-0.5B

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN25_MODEL_PATH = "Qwen/Qwen2.5-0.5b"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(QWEN25_MODEL_PATH)
lm = AutoModelForCausalLM.from_pretrained(QWEN25_MODEL_PATH, dtype=torch.bfloat16).to(device).eval()

prompt = "The weather today is"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    out = lm.generate(ids, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


The weather today is sunny. It's a good day to go for a walk. I'm going to the park.


## Whisper-small

In [3]:
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_MODEL_PATH = "openai/whisper-small"
SAMPLE_AUDIO = "dataset/public/train/0/chunk_0.wav"

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_PATH).to(device).eval()


audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
with torch.no_grad():
    ids = model.generate(feats.to(device), language="en", task="transcribe")
result = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
print(f"Whisper ASR trasncription: {result}")
# for the transcription + decoding pipeline: i'm not exactly certain that on the actual grading machine
# it would run under the 10-minute time limit. if it doesn't and time almost runs out, it can fall back to the 
# baseline solution (the 0.68-0.69 scoring one) for subsequent data points that preserves most of the score

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

Whisper ASR trasncription: I know it's going to save me a lot of money, I hope.


# 3. Solution Using the Language Model and the ASR Model

This pipeline, a **speaker-aware sequential decoding** solution, combines acoustic speaker diarization with language model perplexity to reconstruct sequential conversation flow:

* **Speaker Diarization via MFCCs:** Extracts average MFCC feature vectors for each audio chunk and uses $K$-Means clustering ($k=2$) with `StandardScaler` to separate turn-taking speakers into discrete clusters. Strict data validation rules (e.g., the sizes of the two clusters should be +-1 of each other) are applied to ensure that the alternating speaker constraint (as shown below) can be successfully fulfilled.

* **Whisper Transcription:** Transcribes each audio segment using Whisper to convert speech chunks into raw text representations.

* **Sequential LM Selection:** Starting from the known sequence prefix in `prefix.json`, the pipeline greedily (should probably try beam-search or other techniques for the next step?) selects the next chunk by picking the candidate that minimizes total cross-entropy loss (perplexity) under an autoregressive language model.

* **Alternating Speaker Constraints:** When valid speaker clusters are identified, candidates are constrained to alternate cluster turns (`cur = 1 - cur`) at each step, preventing consecutive predictions from the same speaker and guiding realistic dialogue generation.

In [91]:
import os
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
LB = 'b'
from tqdm import tqdm
dirs = [x for x in os.listdir(f'dataset/private/test_leaderboard_{LB}') if '.' not in x]
#dirs = [x for x in os.listdir(f'dataset/public/train') if '.' not in x]

prefixes = json.load(open(f'dataset/private/test_leaderboard_{LB}/prefix.json'))
#prefixes = json.load(open(f'dataset/public/train/prefix.json'))

answers = {}
cluster_sr = []
for item in tqdm(dirs[:100]):
    entries = f'dataset/private/test_leaderboard_{LB}/' + item
    #entries = f'dataset/public/train/' + item
    audio_files = sorted([entries + '/' + x for x in os.listdir(entries) if ".wav" in x])
    audios = []
    embeds = []
    for fn in audio_files:
        audio, _ = librosa.load(fn, sr=16000)
        audios.append(audio)
        max_len = max(len(a) for a in audios)
        padded_audios = np.array([np.pad(a, (0, max_len - len(a))) for a in audios])
        input_features = processor(padded_audios.tolist(), sampling_rate=16000, return_tensors="pt", padding=True).input_features.to(device)
        embeds.append(librosa.feature.mfcc(y=audio, sr=16000).mean(axis=1))

    km = KMeans(n_clusters=2, n_init=10, random_state=42)
    clusters = km.fit_predict(StandardScaler().fit_transform(embeds))
    #print((clusters == 0).sum(), (clusters == 1).sum())
    with torch.no_grad():
        generated_ids = model.generate(input_features, language="en", task="transcribe")
    results = processor.batch_decode(generated_ids, skip_special_tokens=True)
    temp_results = [r.strip() for r in results]
    cands = list(range(len(audio_files)))
    cands.remove(prefixes[item][0])
    cands.remove(prefixes[item][1])
    most_common = collections.Counter(clusters).most_common(1)[0][0]
    validity = abs(collections.Counter(clusters).most_common(1)[0][1] - collections.Counter(clusters).most_common(2)[1][1]) <= 1
    if (clusters[prefixes[item][0]] + clusters[prefixes[item][1]] == 1) and (len(cands) % 2 == 0 or clusters[prefixes[item][0]] == most_common) and validity: 
        # different clusters (one in cluster 0, one in cluster 1) and the beginning line belongs to the majority cluster
        clustered = True
        #print('clustered!')
        cur = clusters[prefixes[item][0]]
    else:
        clustered = False
    order = [0] * len(audio_files)
    order[prefixes[item][0]] = 0; order[prefixes[item][1]] = 1
    decoded = temp_results[prefixes[item][0]] + " " + temp_results[prefixes[item][1]]
    counter = 2
    cluster_sr.append(clustered)
    #print(clustered)
    while len(cands) > 0:
        record, sel = float('inf'), 0
        for idx, i in enumerate(cands):
            if clustered == False or (clustered == True and int(clusters[i]) == int(cur)):
                s = decoded + " " + temp_results[i]
                inputs = tokenizer(s, return_tensors="pt").to(device)
                with torch.no_grad():
                    perplexity = lm(**inputs, labels=inputs.input_ids).loss
                    if perplexity < record:
                        record = perplexity
                        sel = i
        cur = 1 - cur # flip the current index
        cands.remove(sel)
        order[sel] = counter
        counter += 1
        decoded = decoded + " " + temp_results[sel]
    answers[item] = order

100%|██████████| 100/100 [03:17<00:00,  1.97s/it]


In [92]:
sum(cluster_sr) / len(cluster_sr)

0.75

In [93]:
torch.cuda.empty_cache()
import gc
gc.collect()

417

In [94]:
with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")

wrote answers.json: 100 dialogues


In [95]:
preds = json.load(open('answers.json'))
gt = json.load(open(f'dataset/private/test_leaderboard_{LB}_answers.json'))
#gt = json.load(open('dataset/public/train_answers.json'))
scores = []
for item in preds.keys():
    scores.append(score_one(preds[item], gt[item]))
print(sum(scores) / len(scores))


0.7169484115815385
